In [1]:
# ==========================================
# Cell 1: Mount Drive & Setup Folders
# ==========================================
import os
from google.colab import drive

print("📁 Mounting Google Drive...")
drive.mount('/content/drive')

# Paths setup
PROJECT_DIR = '/content/drive/MyDrive/Video_Translation_Project'
STAGE_1_DIR = os.path.join(PROJECT_DIR, 'Stage_1')
STAGE_2_DIR = os.path.join(PROJECT_DIR, 'Stage_2')
STAGE_3_DIR = os.path.join(PROJECT_DIR, 'Stage_3')
MODELS_CACHE = os.path.join(PROJECT_DIR, 'Models_Cache/LatentSync_Checkpoints')

# Create necessary folders if they don't exist
os.makedirs(STAGE_3_DIR, exist_ok=True)
os.makedirs(os.path.join(MODELS_CACHE, 'whisper'), exist_ok=True)

print("✅ Drive mounted and folders are ready!")

📁 Mounting Google Drive...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Drive mounted and folders are ready!


In [2]:
%%bash
# ==========================================
# Cell 2: Repo Clone & Strict Dependency Fixes
# ==========================================
cd /content
rm -rf LatentSync
echo "⚙️ Cloning LatentSync Repository..."
git clone https://github.com/bytedance/LatentSync.git
cd LatentSync

echo "⚙️ Applying precise version fixes..."
sed -i 's/mediapipe==0.10.11/mediapipe==0.10.14/g' requirements.txt

apt-get update -q
apt-get install -y -q ffmpeg

pip install -q -r requirements.txt

pip install -q huggingface_hub decord kornia insightface ffmpeg-python DeepCache

echo "✅ Strict Environment Setup Complete!"

⚙️ Cloning LatentSync Repository...
⚙️ Applying precise version fixes...
Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:4 https://cli.github.com/packages stable/main amd64 Packages [354 B]
Get:5 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Hit:6 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:7 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [99.9 kB]
Get:8 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,805 kB]
Get:9 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:10 http://security.ubuntu.com/ubuntu jammy-security/main amd64 Packages [4,083 kB]
Get:11 http://security.ubuntu.com/ubuntu jammy-security/restricted amd64 Packages [7,332 kB]
Get:12 https://ppa.launchpadcontent

Cloning into 'LatentSync'...
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
xarray-einstats 0.10.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
google-adk 2.3.0 requires pydantic<3,>=2.12, but you have pydantic 2.11.10 which is incompatible.
google-adk 2.3.0 requires starlette<2,>=1.0.1, but you have starlette 0.52.1 which is incompatible.
tensorflow 2.20.0 requires protobuf>=5.28.0, but you have protobuf 4.25.9 which is incompatible.
python-fasthtml 0.14.3 requires starlette>=1.0.1, but you have starlette 0.52.1 which is incompatible.
google-genai 2.10.0 requires pydantic<3.0.0,>=2.12.5, but you have pydantic 2.11.10 which is incompatible.
shap 0.52

In [8]:
# ==========================================
# Cell 3: Secure Model Downloader (Official ByteDance Repo)
# ==========================================
import os
from huggingface_hub import hf_hub_download

MODELS_CACHE = '/content/drive/MyDrive/Video_Translation_Project/Models_Cache/LatentSync_Checkpoints'
LOCAL_CKPT_DIR = '/content/LatentSync/checkpoints'
os.makedirs(MODELS_CACHE, exist_ok=True)

print("🧹 Checking for and removing corrupted files...")
for filename in ["latentsync_unet.pt", "latentsync_syncnet.pt"]:
    filepath = os.path.join(MODELS_CACHE, filename)
    if os.path.exists(filepath) and os.path.getsize(filepath) < 100_000_000:
        os.remove(filepath)
        print(f"   Deleted corrupted {filename}")

print("📥 Downloading models securely from the OFFICIAL ByteDance repo...")
print("   (This will take a few minutes for the 5GB+ UNet model)")

# 1. Download the main UNet model from the official repo
hf_hub_download(
    repo_id="ByteDance/LatentSync",
    filename="latentsync_unet.pt",
    local_dir=MODELS_CACHE
)

# 2. Download the Whisper model from the official repo
hf_hub_download(
    repo_id="ByteDance/LatentSync",
    filename="whisper/tiny.pt",
    local_dir=MODELS_CACHE
)

print("🔗 Linking folders...")
if os.path.exists(LOCAL_CKPT_DIR):
    os.system(f"rm -rf {LOCAL_CKPT_DIR}")
os.symlink(MODELS_CACHE, LOCAL_CKPT_DIR)

# === VERIFICATION STEP ===
print("\n🔍 Verifying Files...")
expected_file = os.path.join(LOCAL_CKPT_DIR, 'latentsync_unet.pt')
if os.path.exists(expected_file):
    file_size_gb = os.path.getsize(expected_file) / (1024 * 1024 * 1024)
    print(f"✅ Success! latentsync_unet.pt is {file_size_gb:.2f} GB and ready for inference.")
else:
    print(f"❌ ERROR: File missing.")

🧹 Checking for and removing corrupted files...
📥 Downloading models securely from the OFFICIAL ByteDance repo...
   (This will take a few minutes for the 5GB+ UNet model)


latentsync_unet.pt:   0%|          | 0.00/3.40G [00:00<?, ?B/s]

🔗 Linking folders...

🔍 Verifying Files...
✅ Success! latentsync_unet.pt is 3.17 GB and ready for inference.


In [6]:
!pip install --upgrade protobuf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 327.1/327.1 kB 27.7 MB/s eta 0:00:00
  Attempting uninstall: protobuf
    Found existing installation: protobuf 4.25.9
    Uninstalling protobuf-4.25.9:
      Successfully uninstalled protobuf-4.25.9
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
mediapipe 0.10.14 requires protobuf<5,>=4.25.3, but you have protobuf 7.35.1 which is incompatible.
google-ai-generativelanguage 0.6.15 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<6.0.0dev,>=3.20.2, but you have protobuf 7.35.1 which is incompatible.
ydf 0.15.0 requires protobuf<7.0.0,>=5.29.1, but you have protobuf 7.35.1 which is incompatible.
grpcio-status 1.71.2 requires protobuf<6.0dev,>=5.26.1, but you have protobuf 7.35.1 which is incompatible.


In [5]:
!pip install --upgrade accelerate peft

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 29.9 MB/s eta 0:00:00
  Attempting uninstall: accelerate
    Found existing installation: accelerate 0.26.1
    Uninstalling accelerate-0.26.1:
      Successfully uninstalled accelerate-0.26.1


In [9]:
# ==========================================
# Cell 4: Run LatentSync Generation (DEBUG MODE)
# ==========================================
import os
import subprocess

VIDEO_IN = "/content/drive/MyDrive/Video_Translation_Project/Stage_1/Silent_Test.mp4"
AUDIO_IN = "/content/drive/MyDrive/Video_Translation_Project/Stage_2/temp_arabic.wav"
VIDEO_OUT = "/content/drive/MyDrive/Video_Translation_Project/Stage_3/Final_LipSynced_Video.mp4"

CONFIG_PATH = "/content/LatentSync/configs/unet/stage2_efficient.yaml"
CKPT_PATH = "/content/LatentSync/checkpoints/latentsync_unet.pt"

print("🎬 Starting the final video generation process...")

os.chdir("/content/LatentSync")
cmd = [
    "python", "-m", "scripts.inference",
    "--unet_config_path", CONFIG_PATH,
    "--inference_ckpt_path", CKPT_PATH,
    "--video_path", VIDEO_IN,
    "--audio_path", AUDIO_IN,
    "--video_out_path", VIDEO_OUT
]

# Run the command and capture standard error and standard out
process = subprocess.run(cmd, capture_output=True, text=True)

if process.returncode == 0:
    print(f"\n🎉 Success! The video is saved in your Drive at:\n{VIDEO_OUT}")
else:
    print("\n⚠️ An error occurred during the rendering process.")
    print("\n" + "="*40)
    print("🔍 DETAILED ERROR LOG (Scroll to the bottom of this log)")
    print("="*40)
    print(process.stderr)

🎬 Starting the final video generation process...

🎉 Success! The video is saved in your Drive at:
/content/drive/MyDrive/Video_Translation_Project/Stage_3/Final_LipSynced_Video.mp4
